# VS2.0 Collection & ScaNN Index Explorer

> **Why this notebook?**  
> Vertex AI Vector Search 2.0 has **no Cloud Console UI** for collections or indexes.  
> This notebook is your visual dashboard — it lets you inspect every layer of the backend.

## What you will explore

| Section | What you learn |
|---------|---------------|
| 1 | What a **Collection** is — schema, metadata fields, vector fields |
| 2 | How to **list and inspect** the Collection programmatically |
| 3 | What a **ScaNN Index** is and how it works internally |
| 4 | How to verify whether your index **exists and is active** |
| 5 | kNN vs ANN — **search mode detection** |
| 6 | **DataObject count** — how many listings are in the collection |
| 7 | **Live search test** — prove the index is actually being used |
| 8 | Architecture diagram — full picture |

---

## Concept Map

```
Vertex AI Vector Search 2.0
│
└── Collection  (airbnb-listings-collection)
    │
    ├── data_schema     ← JSON Schema: what metadata each DataObject carries
    ├── vector_schema   ← which field is the vector (name + dimensions)
    │
    ├── DataObjects[]   ← the actual data: {id, metadata, vector}
    │                      one per Airbnb listing
    │
    └── Index (optional)
            └── ScaNN ANN index on the 'embedding' field
                  ├── Without index → full kNN scan (exact, slow at scale)
                  └── With index    → ScaNN ANN (~99% recall, sub-10ms)
```

---
## 0 — Setup

In [ ]:
import sys
import json
import time
from pathlib import Path
from datetime import timezone

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Project root → imports config.py ─────────────────────────────────────────
PROJECT_ROOT = Path("../").resolve()
sys.path.insert(0, str(PROJECT_ROOT))
import config

from google.cloud import vectorsearch_v1beta as vs

print(f"Project       : {config.PROJECT_ID}")
print(f"Location      : {config.LOCATION}")
print(f"Collection ID : {config.COLLECTION_ID}")
print(f"Collection    : {config.COLLECTION_RESOURCE}")

In [ ]:
# Two clients — each has a distinct responsibility
vs_client   = vs.VectorSearchServiceClient()      # collections, indexes (schema/config)
do_client   = vs.DataObjectServiceClient()        # create / get / delete DataObjects
search_client = vs.DataObjectSearchServiceClient() # semantic search

COLLECTION = config.COLLECTION_RESOURCE

print("Clients ready:")
print(f"  vs.VectorSearchServiceClient       → manage collections & indexes")
print(f"  vs.DataObjectServiceClient         → CRUD on individual DataObjects")
print(f"  vs.DataObjectSearchServiceClient   → vector similarity search")

---
## Section 1 — What is a Collection?

A **Collection** is the top-level container in VS2.0. Think of it like a database table, but it stores both **metadata** and **vectors** together — no Firestore, no secondary storage.

Every Collection has two schemas defined at creation time:

| Schema | Purpose |
|--------|---------|
| `data_schema` | Declares the metadata fields every DataObject can carry (JSON Schema format) |
| `vector_schema` | Declares the vector field name and number of dimensions |

In [ ]:
# Fetch the Collection object from the API
collection_obj = vs_client.get_collection(
    request=vs.GetCollectionRequest(name=COLLECTION)
)

print("=" * 60)
print("COLLECTION DETAILS")
print("=" * 60)
print(f"  Resource name  : {collection_obj.name}")
print(f"  Display name   : {collection_obj.display_name}")
print(f"  Description    : {collection_obj.description}")
print(f"  Create time    : {collection_obj.create_time}")
print(f"  Update time    : {collection_obj.update_time}")

In [ ]:
# ── data_schema — every metadata field defined at collection creation ──────────
print("\n=== data_schema (metadata fields) ===")
print("These are the fields stored alongside EVERY DataObject in this collection.")
print()

data_schema = collection_obj.data_schema
# data_schema is a google.protobuf.Struct — convert to plain dict
schema_dict = dict(data_schema)
try:
    props = schema_dict.get("properties", {})
    print(f"  {'Field':<25}  {'Type':<10}  Notes")
    print("  " + "-" * 58)
    for field_name, field_def in props.items():
        ftype = field_def.get("type", "?")
        notes = ""
        if field_name == "text":     notes = "← embedding source text (rich description)"
        if field_name == "price":    notes = "← numeric USD/night (parsed from '$97.00')"
        if field_name == "rating":   notes = "← review_scores_rating 0–5"
        if field_name == "instant_bookable": notes = "← 'true'/'false' string"
        print(f"  {field_name:<25}  {ftype:<10}  {notes}")
except Exception:
    print(f"  Raw schema: {schema_dict}")

In [ ]:
# ── vector_schema — the embedding field definition ────────────────────────────
print("\n=== vector_schema (vector field) ===")
print("This declares which field holds the embedding vector and how many dimensions.")
print()

vector_schema = collection_obj.vector_schema
try:
    vs_dict = dict(vector_schema)
    for field_name, field_def in vs_dict.items():
        dims = field_def.get("dense_vector", {}).get("dimensions", "?")
        print(f"  Vector field : '{field_name}'")
        print(f"  Dimensions   : {dims}")
        print(f"  Type         : dense vector (float32)")
        print(f"  Produced by  : text-embedding-005 ({dims}-dim retrieval model)")
except Exception:
    print(f"  Raw schema: {dict(vector_schema)}")

In [ ]:
# Visual: Collection schema diagram
fig, ax = plt.subplots(figsize=(12, 6))
ax.axis("off")

# Outer box — Collection
ax.add_patch(mpatches.FancyBboxPatch((0.01, 0.01), 0.98, 0.97,
    boxstyle="round,pad=0.02", lw=2.5, edgecolor="#4285F4", facecolor="#f0f5ff"))
ax.text(0.5, 0.93, f"VS2.0 Collection:  {config.COLLECTION_ID}",
    ha="center", va="center", fontsize=13, fontweight="bold", color="#4285F4")

# data_schema box
ax.add_patch(mpatches.FancyBboxPatch((0.03, 0.52), 0.44, 0.37,
    boxstyle="round,pad=0.01", lw=1.5, edgecolor="#34A853", facecolor="#f0fff4"))
ax.text(0.25, 0.87, "data_schema", ha="center", va="center",
    fontsize=10, fontweight="bold", color="#34A853")
data_fields = [
    ("STRING", ["name", "neighbourhood", "room_type", "property_type",
                "bathrooms_text", "listing_url", "host_name",
                "host_is_superhost", "instant_bookable", "text"]),
    ("NUMBER", ["accommodates", "bedrooms", "beds", "price",
                "rating", "latitude", "longitude"]),
]
y_pos = 0.82
for dtype, fields in data_fields:
    ax.text(0.06, y_pos, f"{dtype}:", fontsize=8, fontweight="bold",
        color="#333", va="center")
    y_pos -= 0.04
    for f in fields:
        ax.text(0.10, y_pos, f"• {f}", fontsize=7.5, color="#555",
            va="center", family="monospace")
        y_pos -= 0.035
    y_pos -= 0.01

# vector_schema box
ax.add_patch(mpatches.FancyBboxPatch((0.53, 0.52), 0.44, 0.37,
    boxstyle="round,pad=0.01", lw=1.5, edgecolor="#FF385C", facecolor="#fff5f5"))
ax.text(0.75, 0.87, "vector_schema", ha="center", va="center",
    fontsize=10, fontweight="bold", color="#FF385C")
ax.text(0.75, 0.79, '"embedding":', ha="center", va="center",
    fontsize=9, family="monospace", color="#333")
ax.text(0.75, 0.73, "dense_vector", ha="center", va="center",
    fontsize=9, color="#555")
ax.text(0.75, 0.68, "dimensions: 768", ha="center", va="center",
    fontsize=10, fontweight="bold", color="#FF385C")
ax.text(0.75, 0.62, "↑ produced by", ha="center", va="center",
    fontsize=8, color="#888")
ax.text(0.75, 0.57, "text-embedding-005", ha="center", va="center",
    fontsize=8.5, color="#555", style="italic")

# DataObject representation
ax.add_patch(mpatches.FancyBboxPatch((0.03, 0.08), 0.94, 0.37,
    boxstyle="round,pad=0.01", lw=1.5, edgecolor="#9C27B0", facecolor="#f9f0ff"))
ax.text(0.5, 0.42, "DataObjects  (one per Airbnb listing)",
    ha="center", va="center", fontsize=10, fontweight="bold", color="#9C27B0")

for i, (label, color) in enumerate([
    ("id: sha1('airbnb:12345')", "#4285F4"),
    ("data: {name, neighbourhood, price, ...}", "#34A853"),
    ("vectors: {embedding: [0.021, -0.043, ..., 0.017]}  ← 768 floats", "#FF385C"),
]):
    ax.text(0.07, 0.34 - i * 0.08, f"  {label}",
        va="center", fontsize=8.5, family="monospace", color=color)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("VS2.0 Collection — Internal Structure", fontsize=12, pad=10)
plt.tight_layout()
plt.show()

---
## Section 2 — Inspect DataObjects in the Collection

Let's query a few DataObjects directly to see exactly what's stored.

In [ ]:
# List a sample of DataObjects using query_data_objects
# NOTE:
#   - client  : DataObjectSearchServiceClient  (not DataObjectServiceClient)
#   - field   : parent=  (not collection=)
#   - returns : ids + metadata via output_fields
print("Fetching a sample of DataObjects from the collection...")

query_req = vs.QueryDataObjectsRequest(
    parent       = COLLECTION,              # ← correct field name
    page_size    = 5,
    output_fields = vs.OutputFields(        # ← request the metadata fields
        data_fields = [
            "name", "neighbourhood", "room_type",
            "price", "rating", "accommodates",
        ],
    ),
)
query_resp = search_client.query_data_objects(request=query_req)  # ← search_client

sample_objects = list(query_resp.data_objects)
print(f"
✓ Retrieved {len(sample_objects)} sample DataObjects")
print(f"  (next_page_token present: {bool(query_resp.next_page_token)})")

In [ ]:
# Inspect the first DataObject in detail
obj = sample_objects[0]

print("=" * 60)
print("DATAOBJECT — full structure")
print("=" * 60)
print(f"  data_object_id : {obj.data_object_id}")
print(f"  name (resource): {obj.name}")
print()
print("  ── metadata (data fields) ──────────────────────────────")
for k, v in obj.data.items():
    if k == "text":
        print(f"    {k:<25} = {repr(str(v)[:80])}... [{len(str(v))} chars]")
    else:
        print(f"    {k:<25} = {repr(v)}")

print()
print("  ── vectors ─────────────────────────────────────────────")
for vec_name, vec_obj in obj.vectors.items():
    vals = vec_obj.dense.values
    print(f"    field='{vec_name}'  dims={len(vals)}")
    print(f"    first 8 values: {[round(v, 5) for v in vals[:8]]}")
    print(f"    value range   : [{min(vals):.4f}, {max(vals):.4f}]")

In [ ]:
# Summary table of the 5 sample objects
print(f"\nSample DataObjects summary:")
print(f"  {'ID (first 12)':<14}  {'Name':<35}  {'Room type':<20}  {'Price':>8}  {'Rating':>7}")
print("  " + "-" * 95)
for o in sample_objects:
    d     = o.data
    obj_id = o.data_object_id[:12]
    name   = str(d.get("name",  ""))[:33]
    rtype  = str(d.get("room_type", ""))[:18]
    price  = d.get("price", 0)
    rating = d.get("rating", 0)
    vec_dims = len(list(o.vectors.values())[0].dense.values) if o.vectors else 0
    print(f"  {obj_id:<14}  {name:<35}  {rtype:<20}  ${price:>6.0f}  {rating:>6.2f}")

---
## Section 3 — What is a ScaNN Index?

**ScaNN** (Scalable Nearest Neighbors) is Google's high-performance ANN (Approximate Nearest Neighbor) library — the same one that powers Google Search internally.

### Without an index — exact kNN
```
Query vector → compare with EVERY DataObject → sort → return top-K
```
- **100% recall** — always finds the exact nearest neighbors
- **O(N) time** — gets slower as collection grows
- Fine for < ~100k DataObjects

### With a ScaNN index — ANN (Approximate Nearest Neighbor)
```
Query vector → ScaNN index structure → ~99% recall, sub-10ms even at 100M vectors
```
- **~99% recall** — misses ~1% of true nearest neighbors (acceptable for RAG)
- **O(log N) or O(1) time** — stays fast at any scale
- Required for production with > 100k DataObjects

### How ScaNN works internally
```
1. Training phase (happens at index build time):
   All 768-dim vectors → quantized into clusters (Product Quantization)
   Each cluster gets a centroid vector stored in memory

2. Search phase (runtime):
   Query vector → find nearest clusters (coarse search)
               → scan only those clusters (fine search)
               → return top-K with DOT_PRODUCT scores
```

In [ ]:
# Visual: kNN vs ScaNN ANN comparison
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

np.random.seed(42)
N = 120
pts = np.random.randn(N, 2) * 2
query = np.array([1.5, 1.0])

dists = np.linalg.norm(pts - query, axis=1)
top5_idx = np.argsort(dists)[:5]

# ── Left: kNN — scan everything ───────────────────────────────────────────────
ax = axes[0]
ax.scatter(pts[:, 0], pts[:, 1], alpha=0.25, s=20, color="#aaa", zorder=2)
# Show scan lines to all points
for i in range(min(N, 40)):
    ax.plot([query[0], pts[i, 0]], [query[1], pts[i, 1]],
            color="#4285F4", alpha=0.08, lw=0.8, zorder=1)
ax.scatter(pts[top5_idx, 0], pts[top5_idx, 1], s=80, color="#FF385C",
           zorder=4, label="Top-5 results")
ax.scatter(*query, s=160, color="#FF6D00", marker="*", zorder=5, label="Query")
ax.set_title("exact kNN — scans ALL vectors\n(O(N), 100% recall)", fontsize=11)
ax.legend(fontsize=8)
ax.set_xlim(-6, 6)
ax.set_ylim(-6, 6)
ax.set_xlabel("dim 1")
ax.set_ylabel("dim 2")
ax.text(0, -5.5, "→ Correct but slow at scale", ha="center", fontsize=9, color="#666")

# ── Right: ScaNN ANN — only scans relevant clusters ──────────────────────────
ax = axes[1]
# Draw 5 clusters
centers = np.array([[-3, 3], [-3, -2], [0, -3], [3, -2], [2, 2]])
colors  = ["#9E9E9E", "#9E9E9E", "#9E9E9E", "#9E9E9E", "#34A853"]
for ci, (cx, col) in enumerate(zip(centers, colors)):
    cluster_pts = pts[np.argsort(np.linalg.norm(pts - cx, axis=1))[:20]]
    ax.scatter(cluster_pts[:, 0], cluster_pts[:, 1],
               alpha=0.35 if col=="#9E9E9E" else 0.8,
               s=20, color=col, zorder=2)
    circle = plt.Circle(cx, 1.8, fill=False, linestyle="--",
                         edgecolor=col, linewidth=1.2, alpha=0.6)
    ax.add_patch(circle)
    ax.scatter(*cx, s=80, color=col, marker="^", zorder=3)

ax.scatter(pts[top5_idx, 0], pts[top5_idx, 1], s=80, color="#FF385C",
           zorder=5, label="Top-5 results")
ax.scatter(*query, s=160, color="#FF6D00", marker="*", zorder=6, label="Query")
ax.set_title("ScaNN ANN — scans only nearest cluster\n(~99% recall, sub-10ms)", fontsize=11)
ax.legend(fontsize=8)
ax.set_xlim(-6, 6)
ax.set_ylim(-6, 6)
ax.set_xlabel("dim 1")
ax.text(0, -5.5, "→ Approximate but fast at any scale", ha="center", fontsize=9, color="#34A853")

plt.suptitle("kNN vs ScaNN ANN — vector search strategies (2D projection of 768-dim space)",
             fontsize=12)
plt.tight_layout()
plt.show()

---
## Section 4 — Inspect the Index (Is It There? What Does It Look Like?)

In [ ]:
# List all indexes on this collection
print(f"Checking for indexes on collection: {config.COLLECTION_ID}")
print()

indexes = list(
    vs_client.list_indexes(
        request=vs.ListIndexesRequest(parent=COLLECTION)
    )
)

print(f"  Indexes found: {len(indexes)}")
print()

if not indexes:
    print("  ⚠  NO INDEX FOUND")
    print("  Current search mode: exact kNN (full scan)")
    print("  To create a ScaNN ANN index run:")
    print("    python scripts/03_create_index.py")
else:
    print(f"  ✓  {len(indexes)} index(es) found — ANN search is active")
    for idx in indexes:
        print(f"  → {idx.name.split('/')[-1]}")

In [ ]:
# Full index inspection (runs if index exists)
if indexes:
    idx = indexes[0]

    # Decode enum values
    dist_map  = {1: "DOT_PRODUCT", 2: "L2", 3: "COSINE", 0: "UNSPECIFIED"}
    norm_map  = {0: "NONE", 1: "L1", 2: "UNIT_L2_NORM"}

    dist_int = idx.distance_metric if isinstance(idx.distance_metric, int) \
               else int(idx.distance_metric)
    norm_int = idx.dense_scann.feature_norm_type \
               if hasattr(idx, "dense_scann") else 0
    norm_int = norm_int if isinstance(norm_int, int) else int(norm_int)

    print("=" * 60)
    print("INDEX DETAILS")
    print("=" * 60)
    print(f"  Resource name    : {idx.name}")
    print(f"  Index ID         : {idx.name.split('/')[-1]}")
    print(f"  Display name     : {idx.display_name}")
    print(f"  Description      : {idx.description}")
    print(f"  Vector field     : {idx.index_field}")
    print(f"  Distance metric  : {dist_map.get(dist_int, dist_int)}")
    print(f"  Feature norm     : {norm_map.get(norm_int, norm_int)}")
    print(f"  Created          : {idx.create_time}")
    print(f"  Updated          : {idx.update_time}")
    print()
    print("  What these settings mean:")
    print(f"    DOT_PRODUCT + UNIT_L2_NORM = cosine similarity")
    print(f"    (vectors are L2-normalised before indexing, so dot-product")
    print(f"     equals cosine similarity — faster than computing cosine directly)")
else:
    print("No index to inspect. Run scripts/03_create_index.py first.")

In [ ]:
# Visual: Index configuration breakdown
if indexes:
    idx = indexes[0]
    idx_id = idx.name.split('/')[-1]

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.axis("off")

    # Outer box
    ax.add_patch(mpatches.FancyBboxPatch((0.01, 0.01), 0.98, 0.97,
        boxstyle="round,pad=0.02", lw=2, edgecolor="#FF6D00", facecolor="#fff8f0"))
    ax.text(0.5, 0.92, f"ScaNN ANN Index:  {idx_id}",
        ha="center", va="center", fontsize=13, fontweight="bold", color="#FF6D00")

    # Config boxes
    config_items = [
        ("Vector field",     f"'{idx.index_field}'",         "#4285F4", "#e8f0fe",  0.06),
        ("Algorithm",        "ScaNN (Dense ANN)",           "#34A853", "#e6f4ea",  0.30),
        ("Distance metric",  "DOT_PRODUCT",                 "#FF385C", "#fde8ec",  0.54),
        ("Feature norm",     "UNIT_L2_NORM",                "#9C27B0", "#f3e5f5",  0.78),
    ]

    for label, value, edge, face, x in config_items:
        ax.add_patch(mpatches.FancyBboxPatch((x, 0.46), 0.20, 0.35,
            boxstyle="round,pad=0.01", lw=1.5, edgecolor=edge, facecolor=face))
        ax.text(x + 0.10, 0.73, label, ha="center", va="center",
            fontsize=8.5, fontweight="bold", color=edge)
        ax.text(x + 0.10, 0.60, value, ha="center", va="center",
            fontsize=9, family="monospace", color="#333", fontweight="bold")

    # Explanation row
    explanations = [
        ("Which field\ngets indexed",          0.06),
        ("Google's ANN\nlibrary from prod",    0.30),
        ("Similarity\nfunction",               0.54),
        ("L2-normalize vectors\nbefore indexing", 0.78),
    ]
    for note, x in explanations:
        ax.text(x + 0.10, 0.35, note, ha="center", va="center",
            fontsize=7.5, color="#666", multialignment="center")

    # Combined effect
    ax.add_patch(mpatches.FancyBboxPatch((0.06, 0.08), 0.88, 0.20,
        boxstyle="round,pad=0.01", lw=1.5, edgecolor="#555", facecolor="#f5f5f5"))
    ax.text(0.50, 0.21, "DOT_PRODUCT + UNIT_L2_NORM  =  cosine similarity",
        ha="center", va="center", fontsize=10, fontweight="bold", color="#333")
    ax.text(0.50, 0.13,
        "Vectors are L2-normalised before indexing → dot product equals cosine → faster math",
        ha="center", va="center", fontsize=8.5, color="#666")

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.show()
else:
    print("No index found — diagram skipped.")

---
## Section 5 — Search Mode Detection (kNN vs ANN)

VS2.0 **automatically routes** to ScaNN when an index exists. Your `SearchDataObjects` call is identical in both modes — you don't change any code. But you can detect which mode is active at runtime.

In [ ]:
def detect_search_mode(collection_resource: str) -> dict:
    """
    Detect the active search mode for this collection.
    Returns a dict with mode, index_id, and description.
    This is the same logic used in app/rag.py at startup.
    """
    try:
        idxs = list(
            vs_client.list_indexes(
                request=vs.ListIndexesRequest(parent=collection_resource)
            )
        )
        if idxs:
            idx_id = idxs[0].name.split("/")[-1]
            return {
                "mode":        "ANN (ScaNN)",
                "index_id":    idx_id,
                "recall":      "~99%",
                "latency":     "sub-10ms at scale",
                "description": "ScaNN ANN index active — approximate nearest neighbor search",
                "code_change": "None required — VS2.0 routes automatically",
            }
        else:
            return {
                "mode":        "kNN (exact)",
                "index_id":    None,
                "recall":      "100%",
                "latency":     "O(N) — grows with collection size",
                "description": "No index — full scan of all DataObjects",
                "code_change": "None required — same API call",
            }
    except Exception as e:
        return {"mode": "unknown", "error": str(e)}


mode_info = detect_search_mode(COLLECTION)

print("=" * 55)
print("SEARCH MODE DETECTION")
print("=" * 55)
for k, v in mode_info.items():
    print(f"  {k:<15} : {v}")

In [ ]:
# Visual: mode indicator
mode = mode_info["mode"]
is_ann = mode.startswith("ANN")

fig, ax = plt.subplots(figsize=(9, 2.5))
ax.axis("off")

color  = "#34A853" if is_ann else "#FBBC04"
label  = f"✓  Active mode:  {mode}" if is_ann else f"⚠  Active mode:  {mode}"
sub    = mode_info.get("description", "")
detail = f"Recall: {mode_info.get('recall','')}   |   Latency: {mode_info.get('latency','')}"
if is_ann:
    detail += f"   |   Index: {mode_info.get('index_id','')}"

ax.add_patch(mpatches.FancyBboxPatch((0.01, 0.05), 0.98, 0.90,
    boxstyle="round,pad=0.02", lw=2.5, edgecolor=color,
    facecolor=color + "22"))
ax.text(0.5, 0.72, label, ha="center", va="center",
    fontsize=14, fontweight="bold", color=color)
ax.text(0.5, 0.45, sub, ha="center", va="center",
    fontsize=10, color="#444")
ax.text(0.5, 0.22, detail, ha="center", va="center",
    fontsize=9, color="#666")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
# Comparison table: kNN vs ANN
print(f"{'Characteristic':<30}  {'exact kNN (no index)':<28}  {'ScaNN ANN (with index)'}")
print("-" * 85)
rows = [
    ("Algorithm",           "full scan of all vectors",    "Product Quantization + ScaNN"),
    ("Recall",              "100%  (exact)",               "~99%  (approximate)"),
    ("Latency (3k docs)",   "~5–20ms",                    "sub-5ms"),
    ("Latency (100k docs)", "~200–500ms",                 "sub-10ms"),
    ("Latency (1M docs)",   "several seconds",            "sub-10ms"),
    ("Memory",              "none (scans disk/RAM)",      "index loaded in RAM"),
    ("Build time",          "none (immediate)",           "5–20 min (one-time)"),
    ("Code change needed",  "none",                       "none — auto-routed by VS2.0"),
    ("Good for",            "dev / small collections",   "production / > 100k objects"),
]
for r in rows:
    print(f"  {r[0]:<28}  {r[1]:<28}  {r[2]}")

---
## Section 6 — How Many DataObjects Are in the Collection?

VS2.0 has no `COUNT(*)` API. The standard pattern is **page through `query_data_objects`** with a page token until exhausted — but that's slow. For a quick estimate we count pages.

In [ ]:
# Count DataObjects by paginating query_data_objects
# Using a large page_size to minimise API calls
PAGE_SIZE  = 1000
MAX_PAGES  = 20      # safety cap

print(f"Counting DataObjects (page_size={PAGE_SIZE}, max_pages={MAX_PAGES})...")
print("(VS2.0 has no COUNT(*) API — we paginate through query_data_objects)")
print()

total      = 0
page_token = None
pages      = 0
t0         = time.time()

while pages < MAX_PAGES:
    req = vs.QueryDataObjectsRequest(
        parent     = COLLECTION,           # ← correct field name
        page_size  = PAGE_SIZE,
        page_token = page_token or "",
    )
    resp       = search_client.query_data_objects(request=req)  # ← search_client
    batch      = list(resp.data_objects)
    total     += len(batch)
    pages     += 1
    page_token = resp.next_page_token

    print(f"  Page {pages}: +{len(batch)} objects  (running total: {total})")

    if not page_token:
        print(f"  → last page reached")
        break

elapsed = time.time() - t0
print()
print(f"{chr(8212)*40}")
print(f"  Total DataObjects : {total:,}")
print(f"  Pages fetched     : {pages}")
print(f"  Time elapsed      : {elapsed:.1f}s")
if page_token:
    print(f"  ⚠ More objects exist beyond page {MAX_PAGES} — increase MAX_PAGES to count all")

---
## Section 7 — Live Search Test

Let's run an actual semantic search and measure latency. We'll also compare the same query before and after embedding to see how the vector represents meaning.

In [ ]:
from vertexai.language_models import TextEmbeddingModel
import vertexai
vertexai.init(project=config.PROJECT_ID, location=config.LOCATION)

emb_model = TextEmbeddingModel.from_pretrained(config.EMBEDDING_MODEL)
print(f"✓ Embedding model loaded: {config.EMBEDDING_MODEL}")

In [ ]:
# Test with multiple queries to see how the index handles different intents
#
# IMPORTANT: search_data_objects returns ONLY (data_object_id, distance).
# The output_fields param inside VectorSearch is silently ignored.
# To get metadata you must call get_data_object() separately for each hit.
# (This is what app/rag.py does with ThreadPoolExecutor in parallel.)

TEST_QUERIES = [
    "cozy private room near downtown Austin",
    "entire house with pool for family vacation",
    "cheap shared room budget travel",
]

def fetch_metadata(obj_id: str) -> dict:
    """Fetch a DataObject metadata dict by its ID."""
    name = f"{COLLECTION}/dataObjects/{obj_id}"
    obj  = do_client.get_data_object(
        request=vs.GetDataObjectRequest(name=name)
    )
    return dict(obj.data) if obj.data else {}


for query_text in TEST_QUERIES:
    print(f"
{chr(9552)*60}")
    print(f"Query: '{query_text}'")
    print(f"{chr(9472)*60}")

    # Step 1: embed the query
    t0 = time.time()
    [q_result] = emb_model.get_embeddings([query_text])
    q_vec = list(q_result.values)
    embed_ms = (time.time() - t0) * 1000

    # Step 2: vector search — returns only IDs + distances
    req = vs.SearchDataObjectsRequest(
        parent        = COLLECTION,
        vector_search = vs.VectorSearch(
            vector       = vs.DenseVector(values=q_vec),
            search_field = "embedding",
            top_k        = 3,
        ),
    )
    t0 = time.time()
    resp = search_client.search_data_objects(request=req)
    search_ms = (time.time() - t0) * 1000

    # Step 3: fetch metadata for each hit (separate API call per result)
    hits = [(res.data_object.data_object_id, float(res.distance))
            for res in (resp.results or [])]

    print(f"  Embed latency  : {embed_ms:.0f}ms")
    print(f"  Search latency : {search_ms:.0f}ms  ({'ScaNN ANN' if is_ann else 'kNN exact'})")
    print(f"  Hits returned  : {len(hits)} IDs (no metadata yet — fetching...)")
    print()

    for rank, (obj_id, score) in enumerate(hits, 1):
        d = fetch_metadata(obj_id)
        print(f"  [{rank}] score={score:.4f}  {str(d.get('name',obj_id))[:40]}")
        print(f"       {d.get('room_type','')}  |  ")
        print(f"       ${d.get('price',0):.0f}/night  |  ")
        print(f"       {chr(9733)}{d.get('rating',0):.2f}  |  {d.get('neighbourhood','')}")


In [ ]:
# Latency benchmark — run the same search 10 times and plot distribution
BENCHMARK_QUERY = "cozy private room near downtown Austin"
N_RUNS = 10

[q_result] = emb_model.get_embeddings([BENCHMARK_QUERY])
q_vec = list(q_result.values)

latencies = []
for i in range(N_RUNS):
    req = vs.SearchDataObjectsRequest(
        parent        = COLLECTION,
        vector_search = vs.VectorSearch(
            vector        = vs.DenseVector(values=q_vec),
            search_field  = "embedding",
            top_k         = 10,
            output_fields = vs.OutputFields(data_fields=["name", "price"]),
        ),
    )
    t0 = time.time()
    search_client.search_data_objects(request=req)
    latencies.append((time.time() - t0) * 1000)

print(f"Search latency benchmark ({N_RUNS} runs, top_k=10):")
print(f"  Min    : {min(latencies):.1f}ms")
print(f"  Max    : {max(latencies):.1f}ms")
print(f"  Mean   : {np.mean(latencies):.1f}ms")
print(f"  Median : {np.median(latencies):.1f}ms")
print(f"  Mode   : {mode_info['mode']}")

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(1, N_RUNS+1), latencies, color="#4285F4", edgecolor="white")
ax.axhline(np.mean(latencies), color="#FF385C", linestyle="--",
           label=f"mean={np.mean(latencies):.1f}ms")
ax.set_xlabel("Run #")
ax.set_ylabel("Latency (ms)")
ax.set_title(f"SearchDataObjects latency — {mode_info['mode']} — {config.COLLECTION_ID}")
ax.legend()
plt.tight_layout()
plt.show()

---
## Section 8 — Full Architecture Diagram

How Collection, Index, DataObjects, and search clients all fit together.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 8))
ax.axis("off")
ax.set_xlim(0, 13)
ax.set_ylim(0, 8)

def box(ax, x, y, w, h, label, sublabel="", color="#4285F4", face=None):
    face = face or color + "22"
    ax.add_patch(mpatches.FancyBboxPatch((x, y), w, h,
        boxstyle="round,pad=0.12", lw=1.8, edgecolor=color, facecolor=face))
    ax.text(x + w/2, y + h/2 + (0.15 if sublabel else 0),
        label, ha="center", va="center",
        fontsize=9, fontweight="bold", color=color)
    if sublabel:
        ax.text(x + w/2, y + h/2 - 0.22, sublabel, ha="center", va="center",
            fontsize=7.5, color="#555")

def arrow(ax, x1, y1, x2, y2, label="", color="#888"):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
        arrowprops=dict(arrowstyle="->", color=color, lw=1.4))
    if label:
        mx, my = (x1+x2)/2, (y1+y2)/2
        ax.text(mx + 0.05, my, label, fontsize=7.5, color="#555",
            va="center", ha="left")

# ── VS2.0 Collection outer frame ─────────────────────────────────────────────
ax.add_patch(mpatches.FancyBboxPatch((3.8, 0.3), 8.8, 7.3,
    boxstyle="round,pad=0.15", lw=2.5, edgecolor="#4285F4", facecolor="#f0f5ff"))
ax.text(8.2, 7.35, f"VS2.0 Collection: {config.COLLECTION_ID}",
    ha="center", va="center", fontsize=11, fontweight="bold", color="#4285F4")

# ── data_schema ───────────────────────────────────────────────────────────────
box(ax, 4.0, 5.4, 3.8, 1.7, "data_schema",
    "10 string + 7 number fields", "#34A853")
ax.text(4.15, 6.85, "name, neighbourhood, room_type, price, ...",
    fontsize=7, color="#34A853", family="monospace")

# ── vector_schema ─────────────────────────────────────────────────────────────
box(ax, 8.4, 5.4, 3.8, 1.7, "vector_schema",
    "'embedding': dense_vector, 768 dims", "#FF385C")
ax.text(8.55, 6.85, "field='embedding'  dims=768",
    fontsize=7, color="#FF385C", family="monospace")

# ── DataObjects ───────────────────────────────────────────────────────────────
box(ax, 4.0, 3.3, 3.8, 1.7, "DataObjects (3,000)",
    "id + metadata + 768-dim vector", "#9C27B0")

# ── ScaNN Index ───────────────────────────────────────────────────────────────
idx_color = "#FF6D00" if is_ann else "#9E9E9E"
idx_label = "ScaNN ANN Index" if is_ann else "No Index (kNN)"
idx_sub   = "DOT_PRODUCT + UNIT_L2_NORM" if is_ann else "full scan at query time"
box(ax, 8.4, 3.3, 3.8, 1.7, idx_label, idx_sub, idx_color)
if is_ann:
    ax.text(9.8, 3.65, "embedding-ann-index",
        fontsize=7, color=idx_color, family="monospace", ha="center")

# ── Clients (left side) ───────────────────────────────────────────────────────
clients = [
    (0.1, 6.3, "VectorSearch\nServiceClient",   "collections\n& indexes",   "#4285F4"),
    (0.1, 4.4, "DataObject\nServiceClient",     "create/get/\ndelete",      "#9C27B0"),
    (0.1, 2.5, "DataObjectSearch\nServiceClient","semantic\nsearch",        "#FF6D00"),
]
for cx, cy, label, sub, col in clients:
    box(ax, cx, cy, 3.3, 1.4, label, sub, col)

# ── Arrows from clients to collection ────────────────────────────────────────
arrow(ax, 3.4, 7.05, 3.9, 7.05, "get_collection\nlist_indexes", "#4285F4")
arrow(ax, 3.4, 5.15, 3.9, 4.5,  "create_data_object\nbatch_create", "#9C27B0")
arrow(ax, 3.4, 3.25, 3.9, 3.75, "search_data_objects", "#FF6D00")

# ── Internal arrow: DataObjects ↔ Index ──────────────────────────────────────
arrow(ax, 7.8, 4.15, 8.4, 4.15, "indexed by" if is_ann else "scanned by", "#888")

# ── Search flow at bottom ─────────────────────────────────────────────────────
ax.add_patch(mpatches.FancyBboxPatch((0.1, 0.5), 12.5, 2.4,
    boxstyle="round,pad=0.1", lw=1.5, edgecolor="#888", facecolor="#fafafa"))
ax.text(6.35, 2.65, "Search Flow (runtime)",
    ha="center", fontsize=9, fontweight="bold", color="#555")

flow_steps = [
    (0.4,  "User\nquery",      "#333",   "#eee"),
    (2.7,  "text-embedding\n-005  →768d", "#FF385C","#fde8ec"),
    (5.1,  "SearchData\nObjectsRequest", "#FF6D00","#fff3e0"),
    (7.5,  "ScaNN/kNN\nlookup",          "#4285F4","#e8f0fe"),
    (9.9,  "Ranked\nresults",            "#34A853","#e6f4ea"),
]
for i, (fx, fl, fc, ff) in enumerate(flow_steps):
    box(ax, fx, 0.7, 2.1, 1.6, fl, color=fc, face=ff)
    if i < len(flow_steps) - 1:
        arrow(ax, fx + 2.1, 1.5, fx + 2.3, 1.5, color="#aaa")

ax.set_title("VS2.0 Collection + ScaNN Index — Full Architecture",
    fontsize=13, pad=10)
plt.tight_layout()
plt.show()

---
## Section 9 — Quick Reference Cheatsheet

Everything a developer needs to know about VS2.0 backend, in one place.

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════╗
║        VS2.0 Developer Cheatsheet — Collection & Index              ║
╠══════════════════════════════════════════════════════════════════════╣
║  COLLECTION                                                          ║
║  • Top-level container (like a table)                                ║
║  • Defined at creation: data_schema + vector_schema                  ║
║  • Stores DataObjects = { id, metadata (data), vector }             ║
║  • NO UI in Cloud Console — API only                                 ║
║                                                                      ║
║  DATA_SCHEMA                                                         ║
║  • JSON Schema (type: object, properties: {...})                     ║
║  • Field types: 'string' or 'number' only                           ║
║  • Booleans stored as 'true'/'false' strings                        ║
║  • Schema is fixed at creation — cannot add fields later            ║
║                                                                      ║
║  VECTOR_SCHEMA                                                       ║
║  • Declares vector field name and dimensions                         ║
║  • Our field: 'embedding', 768 dims (text-embedding-005)            ║
║                                                                      ║
║  DATAOBJECT                                                          ║
║  • Unit of storage — one per Airbnb listing                         ║
║  • ID: SHA1('airbnb:{listing_id}') — deterministic                  ║
║  • CRUD: DataObjectServiceClient                                     ║
║                                                                      ║
║  SCANN INDEX                                                         ║
║  • Optional ANN index on the 'embedding' field                      ║
║  • Algorithm: ScaNN (Google's production ANN library)               ║
║  • Distance: DOT_PRODUCT + UNIT_L2_NORM = cosine similarity         ║
║  • Build time: 5–20 min (one-time, server-side LRO)                 ║
║  • No code change needed — VS2.0 auto-routes to ScaNN               ║
║                                                                      ║
║  CLIENTS (three, each has a distinct role)                           ║
║  • VectorSearchServiceClient      → manage collections & indexes    ║
║  • DataObjectServiceClient        → CRUD on DataObjects             ║
║  • DataObjectSearchServiceClient  → semantic search                  ║
║                                                                      ║
║  SEARCH REQUEST (same regardless of kNN or ANN mode)                ║
║  SearchDataObjectsRequest(                                           ║
║    parent       = COLLECTION_RESOURCE,                               ║
║    vector_search = VectorSearch(                                     ║
║      vector       = DenseVector(values=[768 floats]),                ║
║      search_field = 'embedding',                                     ║
║      top_k        = N,                                               ║
║      output_fields = OutputFields(data_fields=[...]),                ║
║    )                                                                 ║
║  )                                                                   ║
╚══════════════════════════════════════════════════════════════════════╝
""")

In [ ]:
# Final: print live current state summary
print("=" * 55)
print("LIVE STATE SUMMARY")
print("=" * 55)
print(f"  Collection ID   : {config.COLLECTION_ID}")
print(f"  Project         : {config.PROJECT_ID}")
print(f"  Location        : {config.LOCATION}")
print(f"  DataObjects     : {total:,}  listings ingested")
print(f"  Embedding model : {config.EMBEDDING_MODEL}  (768-dim)")
print(f"  Search mode     : {mode_info['mode']}")
if is_ann:
    print(f"  Index ID        : {mode_info['index_id']}")
    print(f"  ANN recall      : {mode_info['recall']}")
    print(f"  ANN latency     : {mode_info['latency']}")
print("=" * 55)